In [2]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import seaborn as sns

df = pd.read_csv('train.csv')
df.head()

,ID,RevolvingUtilizationOfUnsecuredLines,Age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs
0,9580,0.668999,58,2,0.449504,3425.0,9,1,1,1,1.0,0
1,39755,0.015922,71,0,6.000000,NaN,5,0,0,0,0.0,0
2,118799,0.183062,52,1,0.035593,5000.0,9,0,0,0,0.0,0
3,16489,0.162301,77,0,0.227886,2000.0,8,0,0,0,0.0,0
4,149857,0.404199,30,0,0.026010,5843.0,4,0,0,0,0.0,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier


# 1. Cargar datos
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "SeriousDlqin2yrs"
ID_COL = "ID"

# 2. Separar X, y
X = train.drop(columns=[TARGET])
y = train[TARGET]

# 3. Funciones de preprocesado                              ## hacer matriz de correlacion con el target para ver que columnas merece la pena ser procesadas, si tienen peso o no
def winsorize_series(s, lower_q=0.01, upper_q=0.99):        ## winsorizar: los que pasen del percentil 95 se quedan en 95
    lo = s.quantile(lower_q)                                ## random forest le da igual el escalado, los outliers, te da un feature importance...
    hi = s.quantile(upper_q)
    return s.clip(lo, hi)

def preprocess_df(df, is_train=True, ref_stats=None):
    df = df.copy()

    # Imputación simple
    # MonthlyIncome y NumberOfDependents con mediana
    for col in ["MonthlyIncome", "NumberOfDependents"]:
        if is_train:
            median = df[col].median()
            df[col] = df[col].fillna(median)
        else:
            median = ref_stats["median_" + col]
            df[col] = df[col].fillna(median)

    # Winsorizar variables con outliers fuertes
    outlier_cols = ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]
    if is_train:
        stats = {}
        for col in outlier_cols:
            s_w = winsorize_series(df[col])
            stats["lo_" + col] = df[col].quantile(0.01)
            stats["hi_" + col] = df[col].quantile(0.99)
            df[col] = s_w
        # Guardar medianas
        for col in ["MonthlyIncome", "NumberOfDependents"]:
            stats["median_" + col] = df[col].median()
    else:
        stats = ref_stats
        for col in outlier_cols:
            lo = stats["lo_" + col]
            hi = stats["hi_" + col]
            df[col] = df[col].clip(lo, hi)

    # Eliminar ID del modelo (lo usaremos solo para submission)
    if ID_COL in df.columns:
        df = df.drop(columns=[ID_COL])

    return df, stats if is_train else df


# 4. Preprocesar train y crear validación
X_proc, stats = preprocess_df(X, is_train=True, ref_stats=None)

X_train, X_val, y_train, y_val = train_test_split(
    X_proc, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Definir modelos

# 5.1 Baseline: Regresión logística
logreg_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                n_jobs=-1,
                solver="lbfgs",
            ),
        ),
    ]
)

# 5.2 Modelo principal: XGBoost
xgb_model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=10,  # aproximado para desbalance
    n_jobs=-1,
    random_state=42,
)


# 6. Entrenar y evaluar en validación

# 6.1 LogReg
logreg_pipeline.fit(X_train, y_train)
val_pred_lr = logreg_pipeline.predict_proba(X_val)[:, 1]
auc_lr = roc_auc_score(y_val, val_pred_lr)
print(f"ROC-AUC LogReg: {auc_lr:.4f}")

# 6.2 XGBoost
xgb_model.fit(X_train, y_train)
val_pred_xgb = xgb_model.predict_proba(X_val)[:, 1]
auc_xgb = roc_auc_score(y_val, val_pred_xgb)
print(f"ROC-AUC XGBoost: {auc_xgb:.4f}")

# Elegimos el mejor (probablemente XGBoost)
use_xgb = auc_xgb >= auc_lr
print("Usando modelo:", "XGBoost" if use_xgb else "Logistic Regression")

# 7. Reentrenar en todo el train procesado
if use_xgb:
    final_model = XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=10,
        n_jobs=-1,
        random_state=42,
    )
    final_model.fit(X_proc, y)
else:
    final_model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    n_jobs=-1,
                    solver="lbfgs",
                ),
            ),
        ]
    )
    final_model.fit(X_proc, y)

# 8. Preprocesar test y generar submission
X_test_raw = test.copy()
X_test_proc, _ = preprocess_df(X_test_raw, is_train=False, ref_stats=stats)

test_pred = (
    final_model.predict_proba(X_test_proc)[:, 1]
    if not isinstance(final_model, XGBClassifier)
    else final_model.predict_proba(X_test_proc)[:, 1]
)

submission = pd.DataFrame(
    {
        ID_COL: test[ID_COL],
        TARGET: test_pred,
    }
)

submission.to_csv("submission.csv", index=False)
print("Archivo submission.csv generado.")


c:\Users\oscar\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


ROC-AUC LogReg: 0.8243
ROC-AUC XGBoost: 0.8576
Usando modelo: XGBoost
Archivo submission.csv generado.


In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
import lightgbm as lgb


# ============================
# 1. Cargar datos
# ============================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "SeriousDlqin2yrs"
ID_COL = "ID"

X = train.drop(columns=[TARGET])
y = train[TARGET]

# Preprocesar train
X_proc, stats = preprocess_df(X, is_train=True, ref_stats=None)

X_train, X_val, y_train, y_val = train_test_split(
    X_proc, y, test_size=0.2, random_state=42, stratify=y
)


# ============================
# 2. Modelo XGBoost refinado
# ============================
xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.7,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=10,
    n_jobs=-1,
    random_state=42,
)

xgb_model.fit(X_train, y_train)
pred_xgb = xgb_model.predict_proba(X_val)[:, 1]
auc_xgb = roc_auc_score(y_val, pred_xgb)
print(f"AUC XGBoost: {auc_xgb:.5f}")


# ============================
# 3. Modelo LightGBM refinado
# ============================
lgb_model = lgb.LGBMClassifier(
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=25,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    objective="binary",
    is_unbalance=True,
    random_state=42,
)

lgb_model.fit(X_train, y_train)
pred_lgb = lgb_model.predict_proba(X_val)[:, 1]
auc_lgb = roc_auc_score(y_val, pred_lgb)
print(f"AUC LightGBM: {auc_lgb:.5f}")


# ============================
# 4. Selección del mejor modelo
# ============================
if auc_lgb > auc_xgb:
    best_model = lgb_model
    print("Usando LightGBM como modelo final")
else:
    best_model = xgb_model
    print("Usando XGBoost como modelo final")


# ============================
# 5. Reentrenar en TODO el train
# ============================
best_model.fit(X_proc, y)


# ============================
# 6. Preprocesar test y predecir
# ============================
X_test_proc, _ = preprocess_df(test, is_train=False, ref_stats=stats)

test_pred = best_model.predict_proba(X_test_proc)[:, 1]

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_pred
})

submission.to_csv("submission.csv", index=False)
print("submission.csv generado con éxito.")


AUC XGBoost: 0.85729
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Info] Number of positive: 5587, number of negative: 78413
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,005349 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 964
[LightGBM] [Info